## Graph Intelligence: OpenAI Agents SDK + Neo4j MCP
This notebook demonstrates how to build a generative AI agent that can autonomously interact with a Neo4j Graph Database using the Model Context Protocol (MCP).

### 1. Environment Setup & Configuration
First, we install the necessary libraries.

- **openai-agents**: The OpenAI Agents SDK for building AI agents.  
- **neo4j**: The driver for database connectivity.

In [1]:
!pip install --quiet --upgrade openai-agents neo4j

#### Imports

In [2]:
import base64
import json
import os
import subprocess
from getpass import getpass
from typing import Any

from agents import Agent, Runner, ModelSettings, function_tool
from agents.mcp import MCPServerStreamableHttp
from neo4j import AsyncGraphDatabase

#### LLM Configuration

The OpenAI Agent SDK is provider-agnostic, supporting other LLMs via the LiteLLM integration.

Here we will be using the OpenAI API.

In [3]:
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key")

#### DB Configuration

For this example, we'll use the companies database from the Neo4j demo server, which contains organizations, people, investors, and news articles.

For HTTP transport, you only need to set the `NEO4J_URI` and optionally `NEO4J_DATABASE` (if connecting to a specific database).

In [4]:
os.environ["NEO4J_URI"] = "neo4j+s://demo.neo4jlabs.com"
os.environ["NEO4J_DATABASE"] = "companies"
os.environ["NEO4J_USERNAME"] = "companies"
os.environ["NEO4J_PASSWORD"] = "companies"

### 2. Toolset Configuration

We'll use the official [Neo4j MCP Server](https://github.com/neo4j/mcp) to extend the agent with Neo4j tools. This MCP server provides the agent with capabilities to read the graph schema and execute Cypher queries, enabling it to fetch and analyze data directly from the database.

The following code installs the latest version on Google Colab and similar Linux-based systems. For other operating systems, please consult the [official installation documentation](https://neo4j.com/docs/mcp/current/installation/).

In [19]:
import os

# v1.5.2 has a bug where get-schema requires a dummy `properties: {}` argument,
# causing the agent to receive an error. Pin to v1.5.1 until upstream is fixed.
version = "v1.5.1"
print(f"Installing version: {version}")

# Download the Linux binary
!gh release download {version} --repo neo4j/mcp --pattern "neo4j-mcp_Linux_x86_64.tar.gz" --clobber

# Extract
!tar -xzf neo4j-mcp_Linux_x86_64.tar.gz

# Make executable
!chmod +x neo4j-mcp

# Cleanup
!rm neo4j-mcp_Linux_x86_64.tar.gz

# Install to ~/bin (no sudo needed)
!mkdir -p ~/bin && mv neo4j-mcp ~/bin/
os.environ["PATH"] = os.path.expanduser("~/bin") + ":" + os.environ["PATH"]

# Verify installation
!neo4j-mcp -v


Installing version: v1.5.1
]11;?\

nloading neo4j-mcp_Linux_x86_64.tar.gz ⣾neo4j-mcp version: v1.5.1


This starts a persistent background process that listens for MCP requests.  
**Note:** Authentication is handled via a Base64-encoded `Authorization` header. For HTTP transport, the MCP endpoint is available at `http://localhost:80/mcp` by default.

In [20]:
os.environ["NEO4J_TRANSPORT_MODE"] = "http"
os.environ["NEO4J_MCP_HTTP_PORT"] = "8080"

# Run the server in the background (HTTP mode: don't pass username/password via env - auth is per-request via Basic Auth header)
env = os.environ.copy()
env.pop("NEO4J_USERNAME", None)
env.pop("NEO4J_PASSWORD", None)
subprocess.Popen(["neo4j-mcp"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Encode credentials for HTTP Authorization header
credentials = base64.b64encode(
    f"{os.environ['NEO4J_USERNAME']}:{os.environ['NEO4J_PASSWORD']}".encode()
).decode()
print("MCP server started. Credentials encoded.")


MCP server started. Credentials encoded.


### 3. Agent Construction

We wrap the Agent creation in an async helper function. The agent is provided with the MCP server we configured above, which automatically discovers and exposes all Neo4j tools (schema reading, Cypher execution) to the agent.

Here we create an MCP client that connects to our Neo4j server endpoint using `MCPServerStreamableHttp`. Credentials are passed via the `Authorization` header.

In [22]:
class PatchedMCPServerStreamableHttp(MCPServerStreamableHttp):
    """
    Workaround for a neo4j-mcp server bug where parameter-less tools (get-schema,
    list-gds-procedures) incorrectly declare `properties` as a required argument.
    The SDK validates tool calls locally against the schema, so this patch removes
    the bogus `required` constraint before the schema reaches the agent.
    """

    async def list_tools(self, run_context=None, agent=None):
        tools = await super().list_tools(run_context, agent)
        for tool in tools:
            schema = tool.inputSchema
            props = schema.get("properties", {})
            required = schema.get("required", [])
            # Detect signature: only required field is a generic `properties: {}` object
            if (
                required == ["properties"]
                and "properties" in props
                and props["properties"].get("type") == "object"
                and props["properties"].get("additionalProperties") is True
            ):
                schema["required"] = []
        return tools


def create_mcp_server():
    """Returns a configured MCPServerStreamableHttp instance for the Neo4j MCP server."""
    return PatchedMCPServerStreamableHttp(
        name="Neo4j server",
        params={
            "url": "http://localhost:8080/mcp",
            "headers": {"Authorization": f"Basic {credentials}"},
            "timeout": 10,
        },
    )


async def create_agent(instructions, tools=None, mcp_servers=None):
    """
    Constructs an OpenAI Agent with the given instructions, optional custom tools,
    and optional MCP servers.
    """
    return Agent(
        name="Assistant",
        instructions=instructions,
        mcp_servers=mcp_servers or [],
        tools=tools or [],
        model_settings=ModelSettings(tool_choice="required"),
    )


system_prompt = """Use the MCP tools to answer the questions.
Always use read-schema first to evaluate the graph schema if you want to execute any cypher queries."""

print("Agent factory ready.")


Agent factory ready.


### 4. Execution Engine

The `ask_graph` function orchestrates the conversation, handling tool calls automatically. It also uses `pretty_print` to display the agent's step-by-step reasoning, tool calls, and results.

In [13]:
def pretty_print(result):
    for i, item in enumerate(result.new_items):
        print(f"\n[Step {i + 1}] {item.type}")
        print("-" * 40)

        if item.type == "tool_call_item":
            print(f"  🔧 Tool: {item.raw_item.name}")
            print(f"  📥 Arguments:")
            try:
                args = json.loads(item.raw_item.arguments)
                print(f"     {json.dumps(args, indent=6)}")
            except:
                print(f"     {item.raw_item.arguments}")

        elif item.type == "tool_call_output_item":
            print(f"  📤 Output:")
            try:
                output = json.loads(item.output)
                print(f"     {json.dumps(output, indent=6)}")
            except:
                print(f"     {item.output}")

        elif item.type == "message_output_item":
            print(f"  💬 Message:")
            for content in item.raw_item.content:
                if hasattr(content, "text"):
                    print(f"     {content.text}")


async def ask_graph(agent, query):
    """Runs a query against the agent and pretty-prints the step-by-step result."""
    print(f"Processing: {query}")
    result = await Runner.run(agent, query)
    pretty_print(result)

Now we instantiate the agent with the MCP server and run an example query.

In [23]:
async with create_mcp_server() as server:
    mcp_agent = await create_agent(instructions=system_prompt, mcp_servers=[server])
    await ask_graph(mcp_agent, "How many people are in the database?")

Processing: How many people are in the database?

[Step 1] tool_call_item
----------------------------------------
  🔧 Tool: get-schema
  📥 Arguments:
     {}

[Step 2] tool_call_output_item
----------------------------------------
  📤 Output:
     {'type': 'text', 'text': '[{"key":"_Bloom_Perspective_","value":{"type":"node","properties":{"data":"STRING","id":"STRING","name":"STRING","roles":"LIST","version":"STRING"},"relationships":{"_Bloom_HAS_SCENE_":{"direction":"out","labels":["_Bloom_Scene_"]}}}},{"key":"IndustryCategory","value":{"type":"node","properties":{"id":"STRING","name":"STRING"},"relationships":{"HAS_CATEGORY":{"direction":"in","labels":["Organization"]}}}},{"key":"HAS_CEO","value":{"type":"relationship"}},{"key":"IN_COUNTRY","value":{"type":"relationship"}},{"key":"Fewshot","value":{"type":"node","properties":{"Cypher":"STRING","Question":"STRING","embedding":"LIST","id":"INTEGER"}}},{"key":"Organization","value":{"type":"node","properties":{"diffbotId":"STRING","id"

### 5. Custom Tools

Beyond using existing MCP servers, you can also implement your own custom tools and add them directly to the agent. This allows you to create specialized functionality tailored to your specific use case.

First, we define a Neo4j async driver and an `execute_cypher` helper to run queries.

In [24]:
# Create async driver
driver = AsyncGraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])
)


async def execute_cypher(cypher: str, params: dict) -> list[dict]:
    result = await driver.execute_query(cypher, database_=os.environ["NEO4J_DATABASE"], **params)
    return [record.data() for record in result.records]

The first custom tool uses a fulltext index to search for companies by name. The `@function_tool` decorator automatically generates the tool's name and input schema from the function signature, while the docstring provides the description that the agent uses to understand when and how to call it.

In [25]:
@function_tool
async def find_companies(company_name: str) -> str:
    "List of Companies (company_id, name, summary) by fulltext search"
    query = """
        CALL db.index.fulltext.queryNodes('entity', $search, {limit: 100})
        YIELD node as c, score WHERE c:Organization
        AND NOT EXISTS { (c)<-[:HAS_SUBSIDIARY]-() }
        RETURN c.id as company_id, c.name as name, c.summary as summary LIMIT 10
    """
    try:
        results = await execute_cypher(query, {"search": company_name})
        return json.dumps(results)
    except Exception as e:
        return f"Error: {e}"

The second tool retrieves investors for a given company, returning their IDs, names, and types (e.g., Organization or Person).

In [26]:
@function_tool
async def get_investors(company: str) -> str:
    "Returns the investments by a company by name. Returns list of investment ids, names and types."
    try:
        return await execute_cypher("""
            MATCH (o:Organization)-[:HAS_INVESTOR]->(i)
            WHERE o.name = $company
            RETURN i.id as id, i.name as name, head(labels(i)) as type
        """, {"company": company})
    except Exception as e:
        return f"Error: {e}"

Transforming a Python function into a tool is straightforward — the `@function_tool` decorator handles schema generation automatically. Now we combine the MCP server with our custom tools by passing both to the agent, giving it a unified toolkit spanning MCP-exposed and locally-defined capabilities.

In [27]:
custom_tools = [find_companies, get_investors]
custom_agent_prompt = """Use the MCP tools to answer the questions.
Always use read-schema first to evaluate the graph schema if you want to execute any cypher queries."""

async with create_mcp_server() as server:
    custom_agent = await create_agent(
        instructions=custom_agent_prompt,
        mcp_servers=[server],
        tools=custom_tools,
    )
    await ask_graph(custom_agent, "Which companies has Google invested in?")

Processing: Which companies has Google invested in?

[Step 1] tool_call_item
----------------------------------------
  🔧 Tool: get_investors
  📥 Arguments:
     {
      "company": "Google"
}

[Step 2] tool_call_output_item
----------------------------------------
  📤 Output:
     [{'id': 'ELsv5bECSOiWG_Uhf_txI2w', 'name': 'Ionic Security', 'type': 'Organization'}, {'id': 'EUkm62r-bMOidNtPjTkdVvg', 'name': 'Avere Systems', 'type': 'Organization'}, {'id': 'EX-RLztfkOFqTLoM6xIVnlg', 'name': 'FlexiDAO', 'type': 'Organization'}, {'id': 'EtqXbQ9LaMGq8om4dhYY0Fw', 'name': 'Cloudflare', 'type': 'Organization'}, {'id': 'EWIvDLNCSMCCBYUyz0oFPVQ', 'name': 'Trifacta', 'type': 'Organization'}]

[Step 3] message_output_item
----------------------------------------
  💬 Message:
     Google has invested in the following companies:

1. Ionic Security
2. Avere Systems
3. FlexiDAO
4. Cloudflare
5. Trifacta

If you need more details about any of these companies or their investments, let me know!


## Summary

This notebook demonstrates how to use the **OpenAI Agents SDK** to build conversational agents that interact with a Neo4j graph database. The workflow includes:

1. **Environment Setup** — Installed dependencies, configured credentials, and set up the Neo4j MCP Server for HTTP transport.

2. **Toolset Configuration** — Connected to Neo4j using the official Neo4j MCP Server via HTTP transport, enabling schema reading and Cypher query execution.

3. **Agent Construction** — Wrapped agent creation in a reusable `create_agent` factory and a `create_mcp_server` helper, keeping instantiation clean and consistent.

4. **Execution Engine** — Built a reusable `ask_graph` runner with `pretty_print` to display the agent's step-by-step reasoning, tool calls, and results.

5. **Custom Tools** — Created specialized tools (`find_companies`, `get_investors`) using the `@function_tool` decorator and combined them with the MCP server for a unified agent toolkit.